# Python 入门：数据处理、统计检验与绘图
**适用对象：** 生物信息学零基础研究生  
**运行方式：** 使用 Anaconda Navigator 启动 JupyterLab / Jupyter Notebook。  
**数据文件：** `data/python_stats_example.csv`

本 notebook 练习描述统计、独立样本 t 检验、配对 t 检验、单因素 ANOVA、相关分析和基础可视化。示例数据为教学模拟数据，不代表真实实验结果。

## 0. 导入常用库
先运行下面代码。若提示缺少包，可在 Anaconda Prompt 中安装：`conda install pandas numpy scipy matplotlib seaborn`。

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (6, 4)

## 1. 读取数据与初步检查
建议先检查数据维度、列名、缺失值、重复记录和变量类型。

In [ ]:
from pathlib import Path

data_path = Path("data/python_stats_example.csv")
df = pd.read_csv(data_path)
display(df.head())
print("数据维度：", df.shape)
print("\n数据类型：")
print(df.dtypes)
print("\n缺失值计数：")
print(df.isna().sum())
print("\n描述统计：")
display(df.groupby("group")["measurement"].agg(["count", "mean", "std", "median"]))

## 2. 柱状图、箱线图与散点图
图形应显示数据分布和样本量。对小样本实验，尽量叠加单个观测点，而不只画均值柱状图。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(data=df, x="group", y="measurement", ax=axes[0])
sns.stripplot(data=df, x="group", y="measurement", color="black", alpha=0.65, ax=axes[0])
axes[0].set_title("Distribution by group")
sns.histplot(data=df, x="measurement", hue="group", kde=True, element="step", ax=axes[1])
axes[1].set_title("Measurement distribution")
plt.tight_layout()
plt.show()

## 3. 独立样本 t 检验
独立样本 t 检验用于比较两个相互独立组的均值。这里使用 Welch t 检验（`equal_var=False`），不强制假设两组方差相等。请同时报告均值、标准差、差值和 p 值。

In [ ]:
control = df.loc[df["group"] == "Control", "measurement"].dropna()
treatment = df.loc[df["group"] == "Treatment", "measurement"].dropna()

res = stats.ttest_ind(treatment, control, equal_var=False)
mean_diff = treatment.mean() - control.mean()
print(f"Control: mean={control.mean():.3f}, SD={control.std(ddof=1):.3f}, n={len(control)}")
print(f"Treatment: mean={treatment.mean():.3f}, SD={treatment.std(ddof=1):.3f}, n={len(treatment)}")
print(f"Welch t-test: t={res.statistic:.3f}, p={res.pvalue:.4g}")
print(f"Mean difference (Treatment - Control) = {mean_diff:.3f}")

### 3.1 统计检验的前提与替代方案
- 观测值应相互独立；同一对象重复测量不能当作独立样本。
- t 检验关注均值差异，极小样本下需谨慎检查异常值和分布。
- 若数据严重偏态，可考虑 Mann–Whitney U 检验；它检验的是分布/秩差异，不应简单等同于中位数检验。

In [ ]:
u_res = stats.mannwhitneyu(treatment, control, alternative="two-sided")
print(f"Mann–Whitney U: U={u_res.statistic:.3f}, p={u_res.pvalue:.4g}")

## 4. 配对 t 检验
配对设计需要一一对应的测量，例如同一患者治疗前后。必须按受试者 ID 对齐，检验的是每个对象的差值。

In [ ]:
paired = pd.DataFrame({
    "subject": [f"S{i}" for i in range(1, 9)],
    "before": [9.1, 10.2, 8.7, 11.0, 9.8, 10.5, 8.9, 9.7],
    "after":  [8.7, 9.5, 8.4, 10.1, 9.0, 10.0, 8.3, 9.2]
})
display(paired)
paired_res = stats.ttest_rel(paired["after"], paired["before"])
paired["difference"] = paired["after"] - paired["before"]
print("Mean paired difference:", paired["difference"].mean())
print(f"Paired t-test: t={paired_res.statistic:.3f}, p={paired_res.pvalue:.4g}")

## 5. 单因素方差分析（One-way ANOVA）
用于比较三个及以上独立组的均值。整体 ANOVA 显著只说明至少一组均值不同，若需知道哪些组不同，还需进行事后比较并进行多重检验校正。

In [ ]:
anova_df = pd.DataFrame({
    "group": np.repeat(["Low", "Medium", "High"], 8),
    "value": np.r_[rng.normal(5, 1, 8), rng.normal(6.5, 1, 8), rng.normal(8, 1.2, 8)]
})
display(anova_df.groupby("group")["value"].agg(["mean", "std", "count"]))
sns.boxplot(data=anova_df, x="group", y="value")
sns.stripplot(data=anova_df, x="group", y="value", color="black", alpha=.6)
plt.title("One-way ANOVA example")
plt.show()

groups = [g["value"].to_numpy() for _, g in anova_df.groupby("group")]
a_res = stats.f_oneway(*groups)
print(f"One-way ANOVA: F={a_res.statistic:.3f}, p={a_res.pvalue:.4g}")

## 6. 相关分析
Pearson 相关衡量线性相关；Spearman 相关衡量单调秩相关。相关不代表因果。

In [ ]:
corr_df = pd.DataFrame({
    "gene_x": np.arange(1, 31),
    "gene_y": np.arange(1, 31) * 0.7 + rng.normal(0, 4, 30)
})
pearson = stats.pearsonr(corr_df["gene_x"], corr_df["gene_y"])
spearman = stats.spearmanr(corr_df["gene_x"], corr_df["gene_y"])
print(f"Pearson r={pearson.statistic:.3f}, p={pearson.pvalue:.4g}")
print(f"Spearman rho={spearman.statistic:.3f}, p={spearman.pvalue:.4g}")
sns.regplot(data=corr_df, x="gene_x", y="gene_y")
plt.title("Correlation example")
plt.show()

## 7. 导出图形与结果表
论文绘图通常优先保存为 PDF/SVG 等矢量格式；若期刊要求位图，可保存为 300–600 dpi TIFF/PNG。

In [ ]:
fig, ax = plt.subplots()
sns.boxplot(data=df, x="group", y="measurement", ax=ax)
sns.stripplot(data=df, x="group", y="measurement", color="black", alpha=.65, ax=ax)
fig.tight_layout()
fig.savefig("python_group_comparison.pdf", bbox_inches="tight")
fig.savefig("python_group_comparison.png", dpi=300, bbox_inches="tight")
df.groupby("group")["measurement"].agg(["count", "mean", "std"]).to_csv("group_summary.csv")
print("已导出 python_group_comparison.pdf、python_group_comparison.png 和 group_summary.csv")

## 8. 练习题
1. 将独立样本 t 检验改为单侧检验，并解释何时不适合使用单侧检验。
2. 改变模拟数据的均值差异和样本量，观察 p 值如何变化。
3. 为 ANOVA 增加一个组，并讨论为什么需要事后比较。
4. 把图形标题、坐标轴名称和单位改成适合实验报告的表达。
5. 写出一段结果描述，至少包括各组 n、均值±标准差、检验统计量和 p 值。